# Capítulo 9 · Laboratório 9c: o DC ATR de David Cox
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap09c.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O ATR de 20 dividido pela média exponencial de 20 do fechamento: a mesma régua de volatilidade para ativos diferentes, a estabilidade do indicador, a previsão da volatilidade futura e o tamanho de posição pelo risco numa carteira de 40 ações.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM = "2026-10-01"
CACHE = "dados_cap09c.pkl"
logging.getLogger("yfinance").setLevel(logging.CRITICAL)


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def atr(h, l, c, n=20):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()           # suavização de Wilder


def dc_atr(d, n=20):
    """DC ATR (David Cox): ATR(n) / média exponencial de n do fechamento. Resultado em fração do preço."""
    return atr(d["High"], d["Low"], d["Close"], n) / d["Close"].ewm(span=n, adjust=False).mean()


def limpa(d):
    d = d.dropna()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    if "Volume" in d and (d["Volume"] > 0).mean() > 0.5:      # índices e cripto sem volume confiável ficam de fora do filtro
        ok &= d["Volume"] > 0
    return d[ok]                                               # barra sem negócio (máxima = mínima) zeraria o ATR

## Parte 1: a mesma régua

In [ ]:
ex = ["PETR4.SA", "ITUB4.SA", "^BVSP", "KO", "AAPL", "^GSPC", "BTC-USD", "GC=F"]
if os.path.exists(CACHE):
    E, dados = pd.read_pickle(CACHE)
else:
    E = yf.download(ex, start="2015-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    dados = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    confere({"os exemplos da Parte 1": E, "as 40 ações": dados})
    pd.to_pickle((E, dados), CACHE)
print("Parte 1. ATR(20) em pontos x DC ATR(20) em % do preço, em 30/09/2026")
print("   ativo     ATR em pontos | DC ATR | DC ATR mediano desde 2015 | percentil de hoje")
for t in ex:
    d = limpa(E[t])
    a, dc = atr(d["High"], d["Low"], d["Close"]), dc_atr(d)
    print(f"   {t:<9} {a.iloc[-1]:>13,.2f} | {dc.iloc[-1]:>6.2%} | {dc.median():>25.2%} | {(dc < dc.iloc[-1]).mean():>16.0%}")

## Parte 2: estabilidade e volatilidade futura (Partes 2 e 3)

In [ ]:
D = {t: limpa(dados[t]) for t in TICKERS}
estab, corr_dc, corr_dp = [], [], []
for t in TICKERS:
    d = D[t]
    c = d["Close"]
    dc = dc_atr(d)
    simples = atr(d["High"], d["Low"], c) / c
    estab.append((dc.pct_change().abs().median(), simples.pct_change().abs().median()))
    futura = c.pct_change().rolling(20).std().shift(-20) * np.sqrt(252)   # volatilidade realizada dos 20 pregões seguintes
    dp20 = c.pct_change().rolling(20).std()
    ok = pd.concat([dc, dp20, futura], axis=1).dropna().iloc[::20]          # um ponto a cada 20 pregões
    corr_dc.append(ok.iloc[:, 0].corr(ok.iloc[:, 2]))
    corr_dp.append(ok.iloc[:, 1].corr(ok.iloc[:, 2]))
e = np.array(estab)
print(f"\nParte 2. Variação diária mediana do indicador: DC ATR {np.median(e[:, 0]):.2%} x ATR/fechamento {np.median(e[:, 1]):.2%}")
print(f"\nParte 3. Correlação com a volatilidade dos 20 pregões seguintes (média das 40 ações): "
      f"DC ATR {np.mean(corr_dc):.2f} | desvio padrão de 20 dias {np.mean(corr_dp):.2f}")

## Parte 4: tamanho de posição

In [ ]:
C = pd.DataFrame({t: D[t]["Close"] for t in TICKERS}).loc["2006":]
R = pd.DataFrame({t: D[t]["Close"].pct_change() for t in TICKERS}).reindex(C.index).fillna(0)  # retorno no calendário de cada bolsa
DC = pd.DataFrame({t: dc_atr(D[t]) for t in TICKERS}).reindex(C.index).ffill()
fim_mes = C.groupby(C.index.to_period("M")).tail(1).index
existe = C.ffill().notna()                                              # já negociava (feriado não zera o peso)


def carteira(pesos_mes):
    w = pesos_mes.reindex(C.index).ffill().shift(1).fillna(0)          # pesos decididos no fim do mês anterior
    w = w.where(existe, 0)
    w = w.div(w.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return (w * R).sum(axis=1), w


iguais = existe.loc[fim_mes].astype(float)
inv = (1 / DC.loc[fim_mes]).where(existe.loc[fim_mes])
print("\nParte 4. Carteira de 40 ações, rebalanceada no fim de cada mês, 2006 a 30/09/2026 (sem custo)")
print(f"   maior peso de uma ação no inverso ao DC ATR (mediana dos meses): {(inv.div(inv.sum(axis=1), axis=0)).max(axis=1).median():.1%}")
for rot, pm in [("pesos iguais", iguais), ("peso inverso ao DC ATR", inv)]:
    r, w = carteira(pm)
    eq = (1 + r).cumprod()
    anos = len(r) / 252
    contrib = (w * DC.fillna(0)).loc[fim_mes]                            # risco de cada ação ~ peso x DC ATR
    disp = (contrib.std(axis=1) / contrib.mean(axis=1)).median()
    print(f"   {rot:<24} {eq.iloc[-1] ** (1 / anos) - 1:+.1%} ao ano | volatilidade {r.std() * np.sqrt(252):.1%}"
          f" | maior queda {(eq / eq.cummax() - 1).min():.0%} | dispersão do risco entre ações {disp:.2f}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `atr`, `dc_atr` e `limpa` da célula de abertura. Mostre onde está a suavização de Wilder (`alpha = 1 / n`), por que o DC ATR divide o ATR pela média exponencial de 20 do fechamento, e não pelo fechamento do dia, e por que `limpa` tira os dias com a máxima igual à mínima." Confira com a Parte 1, que imprime o ATR(20) e o DC ATR em 30/09/2026: 2.830 pontos e 1,54% no Ibovespa, e o ouro no percentil 92 do próprio histórico.
2. **Adaptar:** "Crie uma célula nova no fim do notebook que baixe [ATIVO], com o ticker do Yahoo Finance, desde 2015, limpe com `limpa` e imprima, como na Parte 1, o ATR em pontos, o DC ATR de hoje, o DC ATR mediano e o percentil de hoje. Não mude as funções `atr` e `dc_atr` nem o período de 20." Antes de olhar o ativo novo, confira se a Parte 1 continua dando os números do livro: 1,54% no Ibovespa, 2,71% no Bitcoin e 0,87% no S&P 500.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 1, inclua o dólar (BRL=X) e o petróleo (BZ=F). Em que percentil está a volatilidade deles hoje?
2. Na Parte 4, tire a função limpa() e veja o que acontece com o maior peso da carteira.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.